# GANForge -- ESRGAN (Standalone Notebook)

**Enhanced perceptual super-resolution**

This is a **self-contained extract** of the ESRGAN section from
the [GANForge Master Notebook](GANForge_Master_Notebook.ipynb) -- every
shared setup utility it needs (imports, device selection, seeding,
`weights_init`, `show_images`, checkpointing, etc.) is included below in
Section 0, so this notebook runs **top to bottom on its own**, in Colab
or locally, without needing any other file.

> **⚠️ Compute note.** See the Section 0 GPU warning below. Heavier
> architectures (Progressive GAN, StyleGAN2, BigGAN, StackGAN, GameGAN)
> include a lightweight from-scratch training loop sized for a single
> GPU (or CPU, slowly) plus, where realistically available, code to load
> strong pretrained weights for inference.

<a id="0"></a>
## 0. Global Setup & Utilities

Everything shared across all twelve GANs lives here: installs, imports,
device selection, seeding, and small helper functions (`weights_init`,
`show_images`, `save_checkpoint`, `Logger`, an `EMA` helper, and a spectral
-norm-aware conv wrapper). Re-using these keeps every later section short
and focused on what makes *that* GAN different.

**🚀 Pro Tip.** In Google Colab: `Runtime -> Change runtime type -> T4 GPU`
before running anything. Everything here auto-detects CUDA and degrades
gracefully to CPU (slow but functional) if none is found.

In [ ]:
# Colab-only: uncomment to install packages not preinstalled in Colab.
# All of these are optional -- every dataset loader below has an offline
# synthetic fallback, so the notebook still runs without them.
# !pip install -q torchaudio datasets soundfile pillow-simd torchmetrics
# !pip install -q pytorch-fid clean-fid


In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
import os
import io
import math
import time
import random
import warnings
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

import torchvision
import torchvision.transforms as T
import torchvision.utils as vutils

import matplotlib.pyplot as plt

try:
    from tqdm.auto import tqdm
except ImportError:
    # Minimal shim so the notebook still runs if tqdm is somehow missing.
    def tqdm(iterable, *args, **kwargs):
        return iterable

warnings.filterwarnings("ignore")  # GAN training is noisy (deprecation spam); keep output readable

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)


In [ ]:
# ---------------------------------------------------------------------------
# Device setup
# ---------------------------------------------------------------------------
# We centralize device selection so every model/tensor in the notebook is
# created consistently. All training loops below check DEVICE.type == "cuda"
# to decide whether to use mixed precision (torch.cuda.amp), which roughly
# halves memory use and often speeds up training 2-3x on modern GPUs.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

if DEVICE.type == "cuda":
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU found -- running on CPU. Reduce EPOCHS/BATCH_SIZE in "
          "every section for reasonable runtimes.")


In [ ]:
# ---------------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------------
# GANs are notoriously sensitive to initialization -- fixing every RNG makes
# results reproducible across reruns, which matters a lot when you are
# debugging "why did my generator collapse this time".
SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


In [ ]:
# ---------------------------------------------------------------------------
# Shared helper functions used across every GAN section below
# ---------------------------------------------------------------------------

def weights_init(m: nn.Module):
    '''DCGAN-style weight init (Radford et al., 2015).

    Why: GAN training is a saddle-point optimization between two networks
    that constantly move each other's loss landscape. Starting both nets
    from small, zero-centered weights (N(0, 0.02)) avoids one network
    dominating early purely due to init scale, which is a common cause of
    early, unrecoverable mode collapse.

    Uses isinstance (NOT a substring match on the class name) because many
    custom container classes in this notebook -- PGAN_ConvBlock,
    ModulatedConv2d, StyleConvBlock -- have "Conv" in their own name; a
    naive `"Conv" in classname` check would incorrectly try to initialize
    THOSE containers directly (they have no `.weight` of their own) instead
    of only their actual nn.Conv2d/nn.Linear/nn.BatchNorm2d leaf children.
    '''
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if getattr(m, "bias", None) is not None:
            nn.init.constant_(m.bias.data, 0)
    elif isinstance(m, (nn.BatchNorm2d, nn.InstanceNorm2d)) and getattr(m, "affine", False):
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


def show_images(images: torch.Tensor, title: str = "", nrow: int = 8, figsize=(8, 8), normalize=True):
    '''Display a batch of image tensors (B, C, H, W) in [-1, 1] or [0, 1] as a grid.'''
    images = images.detach().cpu()
    grid = vutils.make_grid(images, nrow=nrow, normalize=normalize, value_range=(-1, 1) if normalize else None, padding=2)
    plt.figure(figsize=figsize)
    plt.axis("off")
    plt.title(title)
    npimg = grid.numpy()
    if npimg.shape[0] == 1:
        plt.imshow(npimg[0], cmap="gray")
    else:
        plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()


def save_checkpoint(state: dict, path: str):
    '''Persist model/optimizer state so long training runs survive a Colab disconnect.'''
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(state, path)


def load_checkpoint(path: str, map_location=None):
    if not os.path.exists(path):
        return None
    return torch.load(path, map_location=map_location or DEVICE)


class EMA:
    '''Exponential Moving Average of generator weights.

    Why: instantaneous generator weights oscillate a lot during adversarial
    training (they are chasing a moving discriminator). Averaging weights
    over time (as StyleGAN2 and BigGAN both do) gives a much smoother,
    higher-quality generator for *inference*, at zero extra training cost.
    '''
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model: nn.Module):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone()

    def copy_to(self, model: nn.Module):
        model.load_state_dict(self.shadow, strict=True)


class AvgMeter:
    '''Running mean tracker for loss logging.'''
    def __init__(self):
        self.sum, self.n = 0.0, 0

    def update(self, val, k=1):
        self.sum += float(val) * k
        self.n += k

    @property
    def avg(self):
        return self.sum / max(self.n, 1)


def denorm(x: torch.Tensor) -> torch.Tensor:
    '''Map a tensor from [-1, 1] (tanh output range) back to [0, 1] for display/saving.'''
    return (x.clamp(-1, 1) + 1) / 2


CHECKPOINT_DIR = "checkpoints"
SAMPLES_DIR = "samples"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SAMPLES_DIR, exist_ok=True)
print("Shared utilities ready.")


<a id="10"></a>
## ESRGAN -- Enhanced perceptual super-resolution

### 10.1 Problem Statement & Why this GAN

Wang et al. (2018) refined SRGAN's recipe to remove specific, identifiable
artifacts: SRGAN's BatchNorm layers were found to introduce faint
grid-like artifacts when train/test statistics differ even slightly, so
ESRGAN **removes BatchNorm entirely**. Its residual blocks are replaced
with **Residual-in-Residual Dense Blocks (RRDB)** -- densely-connected
convolutions (à la DenseNet) inside a residually-scaled block inside
another residually-scaled block -- for much deeper, more stable feature
reuse. Its discriminator is upgraded from "is this real or fake" to a
**Relativistic average GAN (RaGAN)**: "is the real image *more realistic
than* the fake one, on average" -- a relative, not absolute, judgment.
Together these solve the same problem as SRGAN (single-image
super-resolution) with **measurably fewer artifacts and sharper,
more natural-looking textures**.

### 10.2 Data Collection & Preparation

Same bicubic-degradation (LR, HR) pairing recipe SRGAN uses: HR images
come from STL-10 (falling back to synthetic textures if offline), LR is
a 4x bicubic-downsampled version built on the fly. **Swap in DIV2K
(`eugenesiow/Div2k` on Hugging Face Datasets, or the official DIV2K
download) for a production-quality run.**

In [ ]:
# ---------------------------------------------------------------------------
# (LR, HR) pairing pipeline -- identical recipe to the SRGAN section of the
# GANForge Master Notebook, duplicated here so this notebook is self-
# contained. Constant names keep the "SRGAN_" prefix since they describe
# the shared super-resolution pipeline convention, not anything SRGAN-model-specific.
# ---------------------------------------------------------------------------
SRGAN_HR_SIZE = 96          # STL-10's native resolution -- convenient, no cropping needed
SRGAN_SCALE = 4
SRGAN_LR_SIZE = SRGAN_HR_SIZE // SRGAN_SCALE   # 24x24 input -> 96x96 output
SRGAN_BATCH_SIZE = 16

hr_transform = T.Compose([T.ToTensor(), T.Normalize((0.5,) * 3, (0.5,) * 3)])

try:
    srgan_hr_base = torchvision.datasets.STL10(root="data", split="train", download=True, transform=hr_transform)
    print(f"Loaded real STL-10 (as HR source; production setups should use DIV2K): {len(srgan_hr_base)} images")
except Exception as e:
    print(f"STL-10 download failed ({e}); falling back to synthetic textures.")
    rng = np.random.default_rng(SEED)
    tex = torch.rand(500, 3, SRGAN_HR_SIZE, SRGAN_HR_SIZE) * 2 - 1
    srgan_hr_base = TensorDataset(tex, torch.zeros(500, dtype=torch.long))

class SRDataset(Dataset):
    '''Builds (LR, HR) PAIRS on the fly: HR is the original image, LR is a
    4x bicubic-downsampled version. Bicubic downsampling is the SRGAN
    paper's degradation model -- swap this for a learned/realistic
    degradation model (blur+noise+JPEG) for production-quality real-world
    super-resolution.'''
    def __init__(self, base):
        self.base = base

    def __len__(self):
        return len(self.base)

    def __getitem__(self, i):
        hr, _ = self.base[i]
        lr = F.interpolate(hr.unsqueeze(0), size=SRGAN_LR_SIZE, mode="bicubic", align_corners=False, antialias=True)
        return lr.squeeze(0).clamp(-1, 1), hr

esrgan_dataset = SRDataset(srgan_hr_base)
esrgan_loader = DataLoader(esrgan_dataset, batch_size=SRGAN_BATCH_SIZE, shuffle=True, drop_last=True)
lr_batch_e, hr_batch_e = next(iter(esrgan_loader))
show_images(F.interpolate(lr_batch_e[:8], size=SRGAN_HR_SIZE, mode="nearest"), title="LR inputs (nearest-upsampled for display)", nrow=8, figsize=(12, 2))
show_images(hr_batch_e[:8], title="HR targets", nrow=8, figsize=(12, 2))


### 10.3 Model Architecture

In [ ]:
class ESRGAN_DenseBlock(nn.Module):
    '''DenseNet-style block: each conv sees the CONCATENATION of the block
    input and every previous conv's output, maximizing feature reuse and
    gradient flow. Output is added back with a small residual scale
    (`beta`, not 1.0) -- ESRGAN's "residual scaling" trick, which the
    paper found necessary to keep very deep RRDB stacks stable without
    any normalization layer at all.'''
    def __init__(self, ch=64, growth=32, beta=0.2):
        super().__init__()
        self.beta = beta
        self.conv1 = nn.Conv2d(ch, growth, 3, 1, 1)
        self.conv2 = nn.Conv2d(ch + growth, growth, 3, 1, 1)
        self.conv3 = nn.Conv2d(ch + 2 * growth, growth, 3, 1, 1)
        self.conv4 = nn.Conv2d(ch + 3 * growth, growth, 3, 1, 1)
        self.conv5 = nn.Conv2d(ch + 4 * growth, ch, 3, 1, 1)
        self.act = nn.LeakyReLU(0.2, True)

    def forward(self, x):
        x1 = self.act(self.conv1(x))
        x2 = self.act(self.conv2(torch.cat([x, x1], 1)))
        x3 = self.act(self.conv3(torch.cat([x, x1, x2], 1)))
        x4 = self.act(self.conv4(torch.cat([x, x1, x2, x3], 1)))
        x5 = self.conv5(torch.cat([x, x1, x2, x3, x4], 1))
        return x + self.beta * x5


class RRDB(nn.Module):
    '''Residual-in-Residual Dense Block: three ESRGAN_DenseBlocks chained,
    with an OUTER residual-scaled skip connection around all three -- the
    "residual in residual" in the name. No BatchNorm anywhere in this block.'''
    def __init__(self, ch=64, growth=32, beta=0.2):
        super().__init__()
        self.beta = beta
        self.blocks = nn.Sequential(ESRGAN_DenseBlock(ch, growth, beta),
                                     ESRGAN_DenseBlock(ch, growth, beta),
                                     ESRGAN_DenseBlock(ch, growth, beta))

    def forward(self, x):
        return x + self.beta * self.blocks(x)


class ESRGAN_UpsampleBlock(nn.Module):
    '''Nearest-neighbor upsample + conv (NOT PixelShuffle) -- the ESRGAN
    paper's choice, which it found produces fewer checkerboard artifacts
    than SRGAN's sub-pixel convolution at the cost of slightly more
    compute.'''
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, 1, 1)
        self.act = nn.LeakyReLU(0.2, True)

    def forward(self, x):
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        return self.act(self.conv(x))


class ESRGAN_Generator(nn.Module):
    def __init__(self, in_ch=3, base_ch=64, n_rrdb=6, growth=32, scale=SRGAN_SCALE):
        super().__init__()
        self.entry = nn.Conv2d(in_ch, base_ch, 3, 1, 1)
        self.body = nn.Sequential(*[RRDB(base_ch, growth) for _ in range(n_rrdb)])
        self.body_conv = nn.Conv2d(base_ch, base_ch, 3, 1, 1)
        n_ups = int(math.log2(scale))
        self.upsample = nn.Sequential(*[ESRGAN_UpsampleBlock(base_ch) for _ in range(n_ups)])
        self.hr_conv = nn.Conv2d(base_ch, base_ch, 3, 1, 1)
        self.act = nn.LeakyReLU(0.2, True)
        self.exit = nn.Conv2d(base_ch, 3, 3, 1, 1)

    def forward(self, x):
        entry_feat = self.entry(x)
        body_feat = self.body_conv(self.body(entry_feat))
        feat = entry_feat + body_feat   # global residual around the entire RRDB stack
        feat = self.upsample(feat)
        feat = self.act(self.hr_conv(feat))
        return torch.tanh(self.exit(feat))


class ESRGAN_Discriminator(nn.Module):
    '''Same VGG-style architecture as SRGAN's discriminator (Section 9.3) --
    what differs is not the network but the LOSS computed from its output
    (see 10.4: relativistic average, not absolute, real/fake).'''
    def __init__(self, base_ch=64):
        super().__init__()
        def conv_bn_lrelu(in_c, out_c, stride, bn=True):
            layers = [nn.Conv2d(in_c, out_c, 3, stride, 1)]
            if bn:
                layers.append(nn.BatchNorm2d(out_c))
            layers.append(nn.LeakyReLU(0.2, True))
            return layers
        layers = []
        layers += conv_bn_lrelu(3, base_ch, 1, bn=False)
        layers += conv_bn_lrelu(base_ch, base_ch, 2)
        layers += conv_bn_lrelu(base_ch, base_ch * 2, 1)
        layers += conv_bn_lrelu(base_ch * 2, base_ch * 2, 2)
        layers += conv_bn_lrelu(base_ch * 2, base_ch * 4, 1)
        layers += conv_bn_lrelu(base_ch * 4, base_ch * 4, 2)
        layers += conv_bn_lrelu(base_ch * 4, base_ch * 8, 1)
        layers += conv_bn_lrelu(base_ch * 8, base_ch * 8, 2)
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(6), nn.Flatten(),
            nn.Linear(base_ch * 8 * 6 * 6, 1024), nn.LeakyReLU(0.2, True),
            nn.Linear(1024, 1),
        )

    def forward(self, x):
        return self.classifier(self.features(x)).view(-1)


G_esrgan = ESRGAN_Generator().to(DEVICE)
D_esrgan = ESRGAN_Discriminator().to(DEVICE).apply(weights_init)


### 10.4 Loss Functions & Optimizers

In [ ]:
# ---------------------------------------------------------------------------
# VGG19 perceptual-loss network and the PSNR metric -- generic super
# -resolution infrastructure (identical to the SRGAN section of the
# GANForge Master Notebook), defined locally here so this notebook has no
# dependency on any other file.
# ---------------------------------------------------------------------------
def build_vgg_feature_extractor():
    '''Perceptual ("content") loss network: a pretrained VGG19's early
    conv-block features, frozen. Comparing SR/HR images in THIS feature
    space (rather than raw pixels) rewards texture/structure similarity
    the way a human perceives it, not literal per-pixel agreement.'''
    try:
        vgg = torchvision.models.vgg19(weights=torchvision.models.VGG19_Weights.IMAGENET1K_V1).features[:36].to(DEVICE).eval()
        for p in vgg.parameters():
            p.requires_grad_(False)
        print("Loaded pretrained VGG19 for perceptual loss.")
        return vgg
    except Exception as e:
        print(f"Pretrained VGG19 unavailable ({e}); falling back to pixel-space MSE as the content loss.")
        return None

srgan_vgg = build_vgg_feature_extractor()
srgan_vgg_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(DEVICE)
srgan_vgg_std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(DEVICE)

def psnr(sr, hr):
    '''Peak Signal-to-Noise Ratio in dB -- the standard quantitative metric
    for super-resolution (higher = closer to the ground-truth HR image).'''
    mse = F.mse_loss(denorm(sr), denorm(hr)).item()
    return 10 * math.log10(1.0 / max(mse, 1e-10))

def esrgan_content_loss(sr, hr):
    if srgan_vgg is None:
        return F.l1_loss(sr, hr)
    sr_n = (denorm(sr) - srgan_vgg_mean) / srgan_vgg_std
    hr_n = (denorm(hr) - srgan_vgg_mean) / srgan_vgg_std
    return F.l1_loss(srgan_vgg(sr_n), srgan_vgg(hr_n))   # ESRGAN uses L1 (not L2/MSE) in feature space -- slightly less blur-prone

def esrgan_relativistic_loss(logits_real, logits_fake):
    '''Relativistic average GAN loss (Jolicoeur-Martineau, 2018):
    D judges "is REAL more realistic than the AVERAGE fake" and vice
    versa, instead of an absolute real/fake call. This gives G a gradient
    even from images D would already call "real" in an absolute sense,
    as long as they are less convincing than the current fakes on
    average -- the paper found this measurably sharper than vanilla GAN
    loss for super-resolution specifically.'''
    rel_real = logits_real - logits_fake.mean()
    rel_fake = logits_fake - logits_real.mean()
    loss_d = F.binary_cross_entropy_with_logits(rel_real, torch.ones_like(rel_real)) + \
             F.binary_cross_entropy_with_logits(rel_fake, torch.zeros_like(rel_fake))
    loss_g = F.binary_cross_entropy_with_logits(rel_real, torch.zeros_like(rel_real)) + \
             F.binary_cross_entropy_with_logits(rel_fake, torch.ones_like(rel_fake))
    return loss_d, loss_g

ESRGAN_LAMBDA_PIXEL = 1e-2      # small direct L1 pixel term for extra stability early in training
ESRGAN_LAMBDA_ADV = 5e-3
esrgan_opt_G = torch.optim.Adam(G_esrgan.parameters(), lr=1e-4, betas=(0.9, 0.999))
esrgan_opt_D = torch.optim.Adam(D_esrgan.parameters(), lr=1e-4, betas=(0.9, 0.999))
ESRGAN_EPOCHS = 6


### 10.5 Training Loop

In [ ]:
esrgan_history = []
for epoch in range(ESRGAN_EPOCHS):
    meters = {"G": AvgMeter(), "D": AvgMeter(), "PSNR": AvgMeter()}
    pbar = tqdm(esrgan_loader, desc=f"[ESRGAN] epoch {epoch+1}/{ESRGAN_EPOCHS}")
    for lr, hr in pbar:
        lr, hr = lr.to(DEVICE), hr.to(DEVICE)
        sr = G_esrgan(lr)

        # ---- Discriminator (relativistic) ----
        D_esrgan.zero_grad(set_to_none=True)
        logits_real = D_esrgan(hr)
        logits_fake = D_esrgan(sr.detach())
        loss_d, _ = esrgan_relativistic_loss(logits_real, logits_fake)
        loss_d.backward()
        esrgan_opt_D.step()

        # ---- Generator: pixel + perceptual + relativistic-adversarial ----
        G_esrgan.zero_grad(set_to_none=True)
        logits_real_for_g = D_esrgan(hr).detach()
        logits_fake_for_g = D_esrgan(sr)
        _, loss_adv_g = esrgan_relativistic_loss(logits_real_for_g, logits_fake_for_g)
        loss_pixel = F.l1_loss(sr, hr)
        loss_content = esrgan_content_loss(sr, hr)
        loss_g = ESRGAN_LAMBDA_PIXEL * loss_pixel + loss_content + ESRGAN_LAMBDA_ADV * loss_adv_g
        loss_g.backward()
        esrgan_opt_G.step()

        meters["G"].update(loss_g.item()); meters["D"].update(loss_d.item()); meters["PSNR"].update(psnr(sr, hr))
        pbar.set_postfix(loss_G=f"{loss_g.item():.3f}", loss_D=f"{loss_d.item():.3f}", PSNR=f"{meters['PSNR'].avg:.1f}dB")

    esrgan_history.append((meters["G"].avg, meters["D"].avg, meters["PSNR"].avg))
    with torch.no_grad():
        G_esrgan.eval()
        sample_sr = G_esrgan(lr[:4])
        G_esrgan.train()
    show_images(torch.cat([F.interpolate(lr[:4], size=SRGAN_HR_SIZE, mode="nearest"), sample_sr, hr[:4]]),
                title=f"ESRGAN epoch {epoch+1}: LR (nearest) / SR / HR", nrow=4)
    save_checkpoint({"G": G_esrgan.state_dict(), "D": D_esrgan.state_dict()}, f"{CHECKPOINT_DIR}/esrgan_epoch{epoch+1}.pt")

print("ESRGAN training complete.")


### 10.6 Inference / Generation Function

In [ ]:
@torch.no_grad()
def esrgan_upscale(lr_images: torch.Tensor) -> torch.Tensor:
    G_esrgan.eval()
    sr = G_esrgan(lr_images.to(DEVICE))
    G_esrgan.train()
    return sr

demo_lr_e = lr_batch_e[:4].to(DEVICE)
show_images(esrgan_upscale(demo_lr_e), title="ESRGAN inference: 4x super-resolved", nrow=4)


### 10.7 Visualization & Results

In [ ]:
gs, ds, psnrs = zip(*esrgan_history)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(gs, label="G"); axes[0].plot(ds, label="D"); axes[0].legend(); axes[0].set_title("Losses")
axes[1].plot(psnrs); axes[1].set_title("PSNR (dB)")
plt.show()

# Bicubic upsampling (a naive, non-learned baseline) vs ESRGAN vs ground truth.
with torch.no_grad():
    lr4, hr4 = lr_batch_e[:4].to(DEVICE), hr_batch_e[:4].to(DEVICE)
    bicubic = F.interpolate(lr4, size=SRGAN_HR_SIZE, mode="bicubic", align_corners=False)
    sr_esrgan = esrgan_upscale(lr4)
fig, axes = plt.subplots(4, 3, figsize=(6, 8))
for row in range(4):
    for col, (img, name) in enumerate(zip([bicubic[row], sr_esrgan[row], hr4[row]], ["Bicubic", "ESRGAN", "Ground truth"])):
        axes[row, col].imshow(denorm(img).permute(1, 2, 0).cpu()); axes[row, col].axis("off")
        if row == 0:
            axes[row, col].set_title(name)
plt.tight_layout(); plt.show()
print(f"Bicubic PSNR: {psnr(bicubic, hr4):.2f}dB | ESRGAN PSNR: {psnr(sr_esrgan, hr4):.2f}dB "
      f"(remember: higher PSNR != more realistic, see 10.9)")


### 10.8 User Input → Output Demo

In [ ]:
IMAGE_INDEX = 1

user_lr_e = lr_batch_e[IMAGE_INDEX:IMAGE_INDEX + 1]
user_sr_e = esrgan_upscale(user_lr_e)
show_images(torch.cat([F.interpolate(user_lr_e, size=SRGAN_HR_SIZE, mode="nearest"), user_sr_e]),
            title="User demo: LR input (nearest-upsampled) vs ESRGAN output", nrow=2, figsize=(4, 2))


### 10.9 Key Insights & Limitations of this GAN

- 💡 **Insight.** Every ESRGAN change (RRDB, no-BN, relativistic
  discriminator, L1-not-L2 perceptual loss) targets a *specific, named
  artifact* SRGAN exhibited -- this is a good general lesson for iterating
  on GANs: identify one concrete failure mode at a time rather than
  redesigning the whole system.
- 🚀 **Pro Tip.** The official ESRGAN paper also releases a **PSNR
  -oriented** (no adversarial loss, pure L1) checkpoint and linearly
  interpolates its weights with the GAN-trained checkpoint at inference
  time -- a cheap way to trade sharpness for fewer hallucinated artifacts
  without retraining.
- ⚠️ **Warning.** RRDB's dense concatenations use more memory per layer
  than plain residual blocks -- our lightweight 6-RRDB model is already
  near the practical CPU-training limit; scale `n_rrdb` up only with a GPU.

**This GAN solved the problem of: single-image super-resolution with
fewer artifacts and more natural texture than SRGAN, by removing
BatchNorm, deepening the residual structure (RRDB), and switching to a
relative (not absolute) adversarial judgment.**